## Ingest Dimension Data into Bronze Layer from sql lite (db) file

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F
import tempfile
import sqlite3
import pandas as pd

catalog_name = 'ecommerce'



### Brands

In [ ]:
volume_path = "/Volumes/ecommerce/source_data/raw/ecomm-raw-data/0_data/brands/brands.db"
brand_schema = StructType([
    StructField("brand_code", StringType(),False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True)
])
# Read binary file from Volume
binary_df = spark.read.format("binaryFile").load(volume_path)
file_content = binary_df.select("content").collect()[0][0]

# Write to a temp file (allowed way)
with tempfile.NamedTemporaryFile(delete=False) as tmp:
    tmp.write(file_content)
    temp_path = tmp.name

# Read with sqlite3
conn = sqlite3.connect(temp_path)
pandas_df = pd.read_sql_query("SELECT * FROM brands", conn)
conn.close()

# Convert to Spark
df = spark.createDataFrame(pandas_df, schema=brand_schema)


display(df.head(5))

In [ ]:
volume_path = "/Volumes/ecommerce/source_data/raw/ecomm-raw-data/0_data/category/category.db"
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# Read binary file from Volume
binary_df = spark.read.format("binaryFile").load(volume_path)
file_content = binary_df.select("content").collect()[0][0]

# Write to a temp file (allowed way)
with tempfile.NamedTemporaryFile(delete=False) as tmp:
    tmp.write(file_content)
    temp_path = tmp.name

# Read with sqlite3
conn = sqlite3.connect(temp_path)
pandas_df = pd.read_sql_query("SELECT * FROM category", conn)
conn.close()

# Convert to Spark
df = spark.createDataFrame(pandas_df, schema=category_schema)


display(df.head(5))

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_category_db")

### Products

In [ ]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/ecomm-raw-data/products/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")    

### Customers

In [ ]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

# Load data using the schema defined
raw_data_path ="/Volumes/ecommerce/source_data/raw/ecomm-raw-data/customers/*.csv"

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(customers_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_customers)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")     

### Date

In [ ]:
volume_path = "/Volumes/ecommerce/source_data/raw/ecomm-raw-data/0_data/date/date.db"
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])
# Read binary file from Volume
binary_df = spark.read.format("binaryFile").load(volume_path)
file_content = binary_df.select("content").collect()[0][0]

# Write to a temp file (allowed way)
with tempfile.NamedTemporaryFile(delete=False) as tmp:
    tmp.write(file_content)
    temp_path = tmp.name

# Read with sqlite3
conn = sqlite3.connect(temp_path)
pandas_df = pd.read_sql_query("SELECT * FROM date", conn)
conn.close()

# Convert to Spark
df = spark.createDataFrame(pandas_df, schema=date_schema)


display(df.head(5))

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_date_db")